# Допуск: Why Do Multi-Agent LLM Systems Fail? (MAST)

**Статья:** M. Cemri и др. — <https://arxiv.org/abs/2503.13657v3>.
Задания сверены с версией v3 от 26.10.2025 (NeurIPS 2025, Datasets and Benchmarks).

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — определите, какому понятию статьи соответствует каждый фрагмент;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы через код и статью;
- **расхождение со статьёй** — найдите одно отклонение от описанного авторами метода;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду.

Для отклонения укажите место, объясните, что не так, сошлитесь на раздел, рисунок,
таблицу или приложение статьи и опишите исправление словами. Писать, запускать и отлаживать
код не нужно. Вопросы относятся только к заданиям без отклонений.

Общие импорты:

In [ ]:
import torch

`trace` — полный журнал взаимодействий агентов, включая вызовы и результаты инструментов.
`judge(trace, definitions=..., few_shot=...)` — LLM-аннотатор, возвращающий словарь
`failure_modes` с флагами 0/1 для режимов MAST. `MODES` — упорядоченный список всех 14
идентификаторов; `MAST` — определения, `EXAMPLES` — размеченные примеры.
Функции, которые используются, но не показаны, считайте корректными. Сниппеты моделируют
разметку и анализ, а не реализацию собственной модели из статьи.

## Задание 1. Похожие симптомы

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

Каждая функция создаёт маленький пример наблюдаемого поведения. Сопоставьте `case_a` …
`case_d` с **Loss of Conversation History**, **Conversation Reset**, **Information
Withholding**, **Ignored Other Agent's Input** и укажите идентификатор и категорию MAST
(§4, прил. A, N; рис. 3). Скрытых отклонений в симуляции нет; выбирайте наиболее точный
первичный механизм, показанный полями записи.

In [ ]:
def case_a():
    history = ["Use LogisticRegression instead of missing LightGBM", "Run the reproduction"]
    visible = ["Run the reproduction"]
    return {"history": history, "visible": visible, "action": "Import LightGBM again"}

def case_b():
    history = ["Investigate the failing test", "The root cause is already located"]
    return {"history": history, "visible": history,
            "message": "Hello! What task would you like me to work on?"}

def case_c():
    return {"known_to_phone_agent": "The login username must be a phone number",
            "sent_to_supervisor": "Invalid credentials; send another username"}

def case_d():
    return {"received_from_peer": "Use the phone number, not the email, as username",
            "visible": "Use the phone number, not the email, as username",
            "action": "Login again with the same email address"}

**Вопросы.**

1. Как поля `visible` позволяют различить `case_a` и `case_b`? Почему одинаковое внешнее
   впечатление «агент забыл задачу» недостаточно для диагноза (прил. A.1–A.2, N.12)?
2. Где разорвана передача информации в `case_c` и `case_d`: до сообщения или после его
   получения? Какого факта не хватает в ответе Phone Agent на рис. 3 (§4, FC2)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Сохранить результат аннотатора

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Запись меток одной трассы, полученных LLM-аннотатором (§3.3, прил. C, N).
Найдите одно отклонение.

In [ ]:
def annotate(trace, judge):
    result = judge(trace, definitions=MAST, few_shot=EXAMPLES)
    flags = torch.tensor([result["failure_modes"][mode] for mode in MODES])  # (14,)
    selected = torch.nn.functional.one_hot(flags.argmax(), num_classes=len(MODES))
    return {mode: int(value) for mode, value in zip(MODES, selected.tolist())}

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Высокая accuracy

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Сравнение человека и аннотатора для **одного фиксированного режима** MAST. Матрица
сопряжённости `(2, 2)` содержит количества: строки — метка человека, столбцы — метка
аннотатора, порядок классов `[нет, есть]`. Это учебный пример дисбаланса, а не восстановленная
матрица из табл. 2. Знаменатели в данном примере ненулевые.

In [ ]:
def agreement(confusion):
    c = confusion.to(torch.float64)                     # (2, 2)
    n = c.sum()
    observed = c.diag().sum() / n
    expected = ((c.sum(dim=1) / n) * (c.sum(dim=0) / n)).sum()
    kappa = (observed - expected) / (1 - expected)
    tn, fp, fn, tp = c.flatten()
    f1 = 2 * tp / (2 * tp + fp + fn)
    return float(observed), float(f1), float(kappa)

confusion = torch.tensor([[94, 0], [6, 0]])
metrics = agreement(confusion)

**Вопросы.**

1. Какие accuracy, F1 и κ вернёт код? Что делает аннотатор и почему высокая accuracy
   здесь мало говорит о его полезности (§3.2–3.3, табл. 2)?
2. В статье приведены κ = 0.88, 0.77 и 0.79. Между кем и на каких данных измерено каждое
   из этих согласий? Что меняют few-shot примеры в табл. 2 и почему 94% accuracy нельзя
   подставлять вместо κ (§3.2–3.4)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Получилась ли задача

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Подсчёт task completion rate. В каждой записи `row` есть `trace` и независимый результат
проверки задачи `task_success`: 0/1, полученный тестами или человеком в зависимости от
бенчмарка. `detected_modes(trace)` возвращает список обнаруженных режимов MAST.
Найдите одно отклонение (§1, табл. 1, прил. J.1, табл. 7).

In [ ]:
def completion_rate(rows, detected_modes):
    completed = []
    for row in rows:
        modes = detected_modes(row.trace)
        completed.append(float(len(modes) == 0))
    return 100 * torch.tensor(completed).mean().item()

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Изменить организацию агентов

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Результаты вмешательств из табл. 5: строки — baseline, improved prompt, new topology;
столбцы — AG2 / GSM-Plus с GPT-4, AG2 / GSM-Plus с GPT-4o, ChatDev / ProgramDev-v0,
ChatDev / HumanEval. Внутри каждого столбца базовая LLM не меняется. Найдите один
неверный комментарий (прил. H.1–H.3).

In [ ]:
def intervention_gains():
    accuracy = torch.tensor([[84.75, 84.25, 25.0, 89.6],
                             [89.75, 89.00, 34.4, 90.3],
                             [85.50, 88.83, 40.6, 91.5]])            # (3, 4), percent
    gains = accuracy[1:] - accuracy[0]                              # percentage points
    # A new topology always outperforms an improved prompt, regardless of model and task.
    return gains

**Ваш ответ:**

_Напишите ответ здесь._